# 02 · PREPARE — Diagnóstico ROCC de calidad de datos
Reliable · Original · Comprehensive · Current. Fuente financiera: base atunera Manabí consolidada desde balances SUPERCIAS 2010–2025 (Superintendencia de Compañías, Valores y Seguros, 2026).

In [1]:
import sys, warnings, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from IPython.display import Image, display
from src.data_loader import RAW, diagnostico_rocc
panel = pd.read_csv(RAW/'panel_variables_clave.csv', dtype={'ruc':str})
rep = diagnostico_rocc(panel, 'SUPERCIAS panel bruto')
print(json.dumps(rep, ensure_ascii=False, indent=1, default=str))

{
 "fuente": "SUPERCIAS panel bruto",
 "total_registros": 2983,
 "anios_disponibles": [
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025
 ],
 "n_empresas": 336,
 "pct_nulos": {
  "ingresos_ordinarios": 32.7,
  "costos_gastos_totales": 4.6,
  "utilidad_antes_part_ir": 4.6
 },
 "ROCC": {
  "Reliable": "SUPERCIAS = regulador oficial | Thai Union = empresa cotizada SET | NOAA = agencia federal EE.UU.",
  "Original": "Balances tal como los presentan las compañías; 2 formularios (NIIF jerárquico y casilleros) no comparables código a código",
  "Comprehensive": "Período 2010–2025",
  "Current": "SUPERCIAS: rezago ~4–16 meses | Bangkok: mensual | NOAA: mensual"
 },
 "empresas_serie_completa": 66,
 "pct_series_completas": 19.6,
 "suficiencia_estadistica": {
  "OLS_pooled": "SUFICIENTE",
  "Panel_efectos_fijos": "SUFICIENTE",
  "Variacion_temporal_del_precio": "LIMITADA (16 años): el precio es común a todas las empresa

In [2]:
display(panel.groupby(['anio','grupo']).size().unstack())
display(panel.groupby('anio').tipo_formulario.value_counts().unstack().fillna(0).astype(int))

grupo,Armadores / pesca marítima,Industria: elaboración y conservación de pescado
anio,,
2010,62,29
2011,73,30
2012,90,32
2013,103,35
2014,112,37
2015,123,38
2016,135,40
2017,152,41
2018,170,44


tipo_formulario,NIIF_jerarquico,casilleros
anio,,
2010,0,91
2011,36,67
2012,122,0
2013,138,0
2014,5,144
2015,6,155
2016,6,169
2017,7,186
2018,7,207


In [3]:
print('Cuadre Activo = Pasivo + Patrimonio:', f"{(panel.cuadre_A_P_Pt.abs()<1).mean():.1%}")
print('Ingresos totales = 0:', f"{(panel.ingresos_totales<=0).mean():.1%} de empresa-año")

Cuadre Activo = Pasivo + Patrimonio: 99.5%
Ingresos totales = 0: 29.6% de empresa-año


### Hallazgos ROCC
- **Reliable:** regulador oficial (SUPERCIAS), empresa cotizada (Thai Union, SET), agencia federal (NOAA). ✔
- **Original:** dos formularios no comparables código a código (NIIF jerárquico vs. casilleros 101) → se usan variables homologadas y se mapea el costo de ventas por formulario (501/51; 797; 7991).
- **Comprehensive:** 336 empresas, 2010–2025. **2014** solo reporta utilidad antes de IR en ~9% y costo de ventas en ~7% → márgenes bruto/operativo 2014 excluidos.
- **Current:** balances 2025 ya disponibles; precio a sep-2026; ONI a ago-2026 (JAS).
- **Suficiencia:** muchas empresas, pero el precio es **común a todas** → la variación identificadora son ~15 años. Toda inferencia sobre el efecto precio se trata como evidencia de series cortas.